# Comparaison avec le case study 2 — SRR11396173 (S15_NL)

SRR11396173 = GSM4430473 = **S15_NL** (He *et al.* 2020). C'est un des 8 échantillons du jeu CellChat `data_humanSkin`
(`data/case_study2.rda`), d'où viennent les 7 563 cellules du mosaic du case study 2 (321 cellules `S15_*`).
SRR11396171 (S13_H) et SRR11396174 (S16_NL) n'y sont pas : comparaison cellule à cellule impossible avec eux.

Prérequis (dans cet ordre) :
1. `bash download_SRR11396173.sh` puis `bash run_kb.sh SRR11396173` (WSL)
2. `Rscript export_case_study2_S15.R` → `cache/case_study2_S15/` et `cache/case_study2_meta.csv`

Les valeurs CellChat sont normalisées (log), pas des comptes : on compare à la log-normalisation de nos comptes kb.

In [ ]:
# Working directory = data/case2/preprocessing/ (FASTQ, kb_out/, write/ and cache/ live there, not tracked by Git)
from pathlib import Path
import os
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts' / 'helpers.R').exists())
os.chdir(root / 'data' / 'case2' / 'preprocessing')


In [ ]:
from __future__ import annotations

import re
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc
import anndata as ad
from scipy.io import mmread
from scipy.sparse import csr_matrix
from scipy.stats import spearmanr, pearsonr
from sklearn.metrics import adjusted_rand_score

In [ ]:
sc.settings.verbosity = 3
sc.set_figure_params(dpi=80, facecolor="white")

In [ ]:
SRR = "SRR11396173"
SAMPLE = "S15"
kb_root = f"kb_out/{SRR}"
cc_dir = f"cache/case_study2_{SAMPLE}"
results_file = f"write/{SRR}.h5ad"
mosaic_file = "../causalccc_server_w0788tM1wzXGHLzNaKKa/input.tsv"
# gènes renommés entre l'annotation de CellChat et celle de kb (GENCODE récent)
ALIASES = {"ATP5E": "ATP5F1E", "H3F3B": "H3-3B", "SELM": "SELENOM"}

## 1. Chargement

In [ ]:
def read_kb(kind: str) -> ad.AnnData:
    d = f"{kb_root}/counts_{kind}"
    X = csr_matrix(mmread(f"{d}/cells_x_genes.mtx")).astype("float32")
    barcodes = pd.read_csv(f"{d}/cells_x_genes.barcodes.txt", header=None)[0].values
    gene_ids = pd.read_csv(f"{d}/cells_x_genes.genes.txt", header=None)[0].values
    gene_names = pd.read_csv(f"{d}/cells_x_genes.genes.names.txt", header=None)[0].fillna(pd.Series(gene_ids)).values
    a = ad.AnnData(X=X, obs=pd.DataFrame(index=barcodes), var=pd.DataFrame({"gene_ids": gene_ids}, index=gene_names))
    a.var_names_make_unique()
    a.obs["sample"] = SRR
    return a

adata = read_kb("filtered")
unf = read_kb("unfiltered")
adata, unf.shape

In [ ]:
# CellChat, échantillon S15 : cellules x gènes
cc = ad.AnnData(
    X=csr_matrix(mmread(f"{cc_dir}/matrix.mtx")).T.tocsr().astype("float32"),
    obs=pd.read_csv(f"{cc_dir}/meta.csv").set_index("barcode"),
    var=pd.DataFrame(index=pd.read_csv(f"{cc_dir}/genes.txt", header=None)[0].values),
)
cc.var_names_make_unique()
cc.var["kb_name"] = [ALIASES.get(g, g) for g in cc.var_names]
cc.obs["labels"].value_counts()

## 2. Recouvrement des barcodes

In [ ]:
unf_umi = pd.Series(np.asarray(unf.X.sum(axis=1)).ravel(), index=unf.obs_names)
bc = cc.obs_names
in_filt = bc.isin(adata.obs_names)
in_unf = bc.isin(unf.obs_names)
print(f"Cellules CellChat {SAMPLE} : {len(bc)}")
print(f"  dans counts_filtered   : {in_filt.sum()} ({in_filt.mean():.0%})")
print(f"  dans counts_unfiltered : {in_unf.sum()} ({in_unf.mean():.0%})")
print(f"  UMI kb médian de ces cellules : {unf_umi[bc[in_unf]].median():.0f}"
      f"  (cellules kb filtered : {unf_umi[adata.obs_names].median():.0f})")
print("Cellules CellChat absentes de counts_filtered, par type :")
print(cc.obs.loc[~in_filt, "labels"].value_counts())

In [ ]:
# rang (knee plot) des cellules CellChat parmi tous les barcodes kb
rank = unf_umi.sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(6, 4))
ax.loglog(np.arange(1, len(rank) + 1), rank.values, color="lightgrey", label="tous les barcodes")
pos = pd.Series(np.arange(1, len(rank) + 1), index=rank.index)
hit = bc[in_unf]
ax.scatter(pos[hit], rank[hit], s=6, color="C3", label=f"cellules CellChat {SAMPLE}")
ax.axhline(unf_umi[adata.obs_names].min(), ls="--", color="k", lw=0.8, label="seuil kb filtered")
ax.set_xlabel("rang du barcode"); ax.set_ylabel("UMI"); ax.legend();

## 3. Preprocessing (mêmes paramètres que `preprocessing_clustering_SRR11396171.ipynb`)

In [ ]:
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)
adata.var["mt"] = adata.var_names.str.startswith("MT-")
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], percent_top=None, log1p=False, inplace=True)
adata.obs["in_cellchat"] = pd.Categorical(np.where(adata.obs_names.isin(bc), "oui", "non"))
sc.pl.violin(adata, ["n_genes_by_counts", "total_counts", "pct_counts_mt"], groupby="in_cellchat", multi_panel=True)

In [ ]:
qc_pass = (adata.obs.n_genes_by_counts > 200) & (adata.obs.n_genes_by_counts < 6000) & (adata.obs.pct_counts_mt < 15)
print(pd.crosstab(adata.obs["in_cellchat"], qc_pass, rownames=["dans CellChat"], colnames=["passe notre QC"]))
adata = adata[qc_pass, :].copy()
adata.layers["counts"] = adata.X.copy()
sc.pp.scrublet(adata)
print(pd.crosstab(adata.obs["in_cellchat"], adata.obs["predicted_doublet"], rownames=["dans CellChat"], colnames=["doublet"]))
adata = adata[~adata.obs["predicted_doublet"]].copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata

## 4. Expression : cellules communes

Pour chaque cellule commune, corrélation (Spearman) entre notre log-normalisation et la valeur CellChat sur les gènes communs ;
pour chaque gène, corrélation (Pearson) entre cellules.

In [ ]:
common_cells = adata.obs_names.intersection(cc.obs_names)
cc_genes = cc.var.reset_index().drop_duplicates("kb_name").set_index("kb_name")["index"]
common_genes = adata.var_names.intersection(cc_genes.index)
print(f"{len(common_cells)} cellules communes, {len(common_genes)} gènes communes "
      f"(sur {cc.n_vars} gènes CellChat, {adata.n_vars} gènes kb)")

ours = adata[common_cells, common_genes].X.toarray()
theirs = cc[common_cells, cc_genes[common_genes].values].X.toarray()

per_cell = np.array([spearmanr(ours[i], theirs[i]).statistic for i in range(len(common_cells))])
expressed = (theirs > 0).sum(axis=0) >= 10
per_gene = pd.Series(
    [pearsonr(ours[:, j], theirs[:, j]).statistic for j in np.where(expressed)[0]],
    index=common_genes[expressed],
)
print(f"Spearman par cellule : médiane {np.median(per_cell):.3f} (min {per_cell.min():.3f})")
print(f"Pearson par gène (≥10 cellules exprimantes, n={expressed.sum()}) : médiane {per_gene.median():.3f}")

In [ ]:
n_genes_ours = (ours > 0).sum(axis=1)
n_genes_theirs = (theirs > 0).sum(axis=1)
fig, axs = plt.subplots(1, 3, figsize=(14, 4), layout="constrained")
axs[0].hist(per_cell, bins=40); axs[0].set_title("Spearman par cellule")
axs[1].hist(per_gene.dropna(), bins=40); axs[1].set_title("Pearson par gène")
axs[2].scatter(n_genes_theirs, n_genes_ours, s=6)
lim = max(n_genes_ours.max(), n_genes_theirs.max())
axs[2].plot([0, lim], [0, lim], "k--", lw=0.8)
axs[2].set_xlabel("gènes détectés (CellChat)"); axs[2].set_ylabel("gènes détectés (kb)");

In [ ]:
# gènes les moins concordants (candidats : différences d'annotation, multi-mapping kallisto vs CellRanger)
per_gene.sort_values().head(20)

## 5. Clustering et labels CellChat

In [ ]:
sc.pp.highly_variable_genes(adata, layer="counts", n_top_genes=2000, flavor="seurat_v3")
adata.layers["scaled"] = adata.X.toarray()
sc.pp.regress_out(adata, ["total_counts", "pct_counts_mt"], layer="scaled")
sc.pp.scale(adata, max_value=10, layer="scaled")
sc.pp.pca(adata, layer="scaled", svd_solver="arpack", mask_var="highly_variable")
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30)
sc.tl.umap(adata)
sc.tl.leiden(adata, resolution=0.7, random_state=0, flavor="igraph", n_iterations=2, directed=False)
adata.obs["cellchat_label"] = cc.obs["labels"].reindex(adata.obs_names).astype("category")
sc.pl.umap(adata, color=["leiden", "cellchat_label"], wspace=0.4)

In [ ]:
lab = adata.obs.loc[common_cells]
print(f"ARI (leiden vs labels CellChat, {len(common_cells)} cellules) : "
      f"{adjusted_rand_score(lab['leiden'], lab['cellchat_label']):.3f}")
pd.crosstab(lab["leiden"], lab["cellchat_label"])

In [ ]:
marker_genes = {
    "Fibroblasts": ["COL1A1", "DCN", "LUM", "APOE", "FBN1"],
    "T cells": ["CD3D", "PTPRC", "CD40LG"],
    "DCs": ["CD1C", "CD74", "HLA-DRA", "CD207"],
}
marker_genes = {k: [g for g in v if g in adata.var_names] for k, v in marker_genes.items()}
sc.pl.dotplot(adata, marker_genes, groupby="leiden", standard_scale="var")

## 6. Gènes du mosaic du case study 2

In [ ]:
header = pd.read_csv(mosaic_file, sep="\t", nrows=0).columns
mosaic_genes = sorted({re.sub(r"(_(DCs|T_cell|Fibroblasts))+$", "", c) for c in header
                       if not c.startswith(("labels", "condition"))})
kb_names = [ALIASES.get(g, g) for g in mosaic_genes]
present = [g in adata.var_names for g in kb_names]
print(f"{sum(present)}/{len(mosaic_genes)} gènes du mosaic présents après notre QC")
print("absents :", [g for g, p in zip(mosaic_genes, present) if not p])
mosaic_corr = per_gene.reindex([g for g in kb_names if g in per_gene.index])
print(f"Pearson par gène sur les gènes du mosaic : médiane {mosaic_corr.median():.3f}")
mosaic_corr.sort_values().head(10)

In [ ]:
adata.write(results_file, compression="gzip")